# Proyecto de Análisis númerico
## Uso de la FFT para procesamiento de imágenes

**Tema seleccionado:** detección y análisis de desenfoque en imágenes usando la Transformada Rápida de Fourier en dos dimensiones.

**Objetivo general:** aplicar la FFT 2D para transformar una imagen del dominio espacial al dominio de frecuencias, visualizar su espectro, estudiar cómo el desenfoque afecta las frecuencias altas y construir un detector básico de desenfoque.

**Preguntas guía del cuaderno:**

1. ¿Cómo se representa una imagen en el dominio de frecuencias?
2. ¿Qué ocurre con el espectro de Fourier cuando una imagen se suaviza o se desenfoca?
3. ¿Se puede usar la energía de frecuencias altas para clasificar una imagen como borrosa o no borrosa?
4. ¿Cómo afecta el parámetro de umbral a la clasificación?

## 1. Fundamento matemático mínimo

Una imagen en escala de grises puede modelarse como una matriz $f(x,y)$ de tamaño $M \times N$, donde cada entrada representa la intensidad de un píxel.

La Transformada Discreta de Fourier bidimensional se define como:

$$
F(u,v)=\sum_{x=0}^{M-1}\sum_{y=0}^{N-1} f(x,y)
e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$

La transformada inversa permite regresar al dominio espacial:

$$
f(x,y)=\frac{1}{MN}\sum_{u=0}^{M-1}\sum_{v=0}^{N-1} F(u,v)
e^{j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$

En imágenes:

- Las **bajas frecuencias** representan zonas suaves o cambios lentos de intensidad.
- Las **altas frecuencias** representan bordes, detalles finos, textura y ruido.
- Una imagen borrosa pierde detalles; por tanto, su energía en altas frecuencias disminuye.

## 2. Librerías necesarias

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy import ndimage
from numpy.fft import fft2, ifft2, fftshift, ifftshift

plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["image.cmap"] = "gray"

## 3. Cargar o generar una imagen de prueba

Coloca una imagen propia dentro de una carpeta llamada `data` con el nombre `imagen.jpg`, `imagen.png` o cambia la ruta en la variable `ruta_imagen`.

Si no se encuentra una imagen externa, el cuaderno usa una imagen de ejemplo de `skimage`. Si `skimage` tampoco está instalado, genera una imagen sintética con patrones geométricos.

In [ ]:
def normalizar_imagen(img):
    # Convierte una imagen a flotante en el intervalo [0, 1].
    img = np.asarray(img, dtype=float)
    if img.ndim == 3:
        # Conversión simple RGB -> escala de grises
        img = 0.2989*img[..., 0] + 0.5870*img[..., 1] + 0.1140*img[..., 2]
    img = img - img.min()
    if img.max() > 0:
        img = img / img.max()
    return img

ruta_imagen = "data/Imagen_nitida.jpeg"

if os.path.exists(ruta_imagen):
    from PIL import Image
    imagen = normalizar_imagen(Image.open(ruta_imagen))
else:
    # Si no se encuentra la imagen, se intenta cargar una imagen de ejemplo de skimage. 
    try:
        from skimage import data
        imagen = normalizar_imagen(data.camera())
    except Exception:
        # Imagen sintética: cuadros, líneas y círculos para producir altas frecuencias
        n = 256
        y, x = np.ogrid[:n, :n]
        imagen = np.zeros((n, n), dtype=float)
        imagen[40:90, 40:210] = 1
        imagen[120:190, 30:100] = 0.7
        imagen[:, ::16] = 0.5
        imagen[::16, :] = 0.5
        mascara = (x - 180)**2 + (y - 160)**2 < 35**2
        imagen[mascara] = 1

plt.imshow(imagen)
plt.title("Imagen original en escala de grises")
plt.axis("off")
plt.show()

print("Dimensiones de la imagen:", imagen.shape)
print("Valor mínimo:", imagen.min(), "Valor máximo:", imagen.max())